<a href="https://colab.research.google.com/github/Patro331/blood-smear-sickle-cell-classification/blob/main/notebooks/03_preprocessing_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Blood Smear Sickle Cell Classification
## MSB7215: Machine Learning in Biomedicine
### Notebook 03: Preprocessing Pipeline

**Author:** Okidi Patrovas Gabriel | 2025/HD07/26020U
**Institution:** Makerere University, Kampala, Uganda

## Overview

This notebook builds the preprocessing pipeline for the HOG
features matrix extracted in the previous notebook.

The pipeline follows standard supervised machine learning practice:

1. Load the extracted features matrix and target vector
2. Split into train and test sets before any preprocessing
3. Build a numeric preprocessing pipeline with StandardScaler
4. Apply the pipeline to training data and transform test data

The train test split is performed before fitting the preprocessor
to prevent data leakage. Fitting the scaler on the full dataset
before splitting would allow information from the test set to
influence the preprocessing, producing overly optimistic results
that do not reflect real-world performance.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

print("All libraries imported successfully")

All libraries imported successfully


## 1. Load Extracted Features

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/sickle-cell-detection'
SAVE_DIR = f'{BASE_DIR}/data/ml_features'
FIG_DIR  = f'{BASE_DIR}/figures'

# Load features matrix and target vector
X         = np.load(f'{SAVE_DIR}/X_features.npy')
y         = np.load(f'{SAVE_DIR}/y_labels.npy')
filenames = np.load(f'{SAVE_DIR}/filenames.npy')

print(f"Features matrix shape: {X.shape}")
print(f"Target vector shape:   {y.shape}")
print(f"Positive samples:      {(y==1).sum()}")
print(f"Negative samples:      {(y==0).sum()}")
print(f"\nFirst 5 feature values of first image:")
print(np.round(X[0, :5], 4))

Mounted at /content/drive
Features matrix shape: (933, 8100)
Target vector shape:   (933,)
Positive samples:      422
Negative samples:      511

First 5 feature values of first image:
[0.0661 0.2661 0.3963 0.0583 0.0908]


### Observations

The features matrix and target vector loaded correctly. We have
933 samples and 8100 features per sample, matching the output
from the feature extraction notebook. The class distribution
is preserved with 422 positive and 511 negative samples.

In [ ]:
# Train test split — done BEFORE preprocessing
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Train Test Split Results:")
print(f"  Training set:   {X_train.shape[0]} samples")
print(f"  Test set:       {X_test.shape[0]} samples")
print(f"  Split ratio:    80% train / 20% test")

print(f"\nClass distribution in training set:")
print(f"  Positive (1):  {(y_train==1).sum()} ({(y_train==1).sum()/len(y_train)*100:.1f}%)")
print(f"  Negative (0):  {(y_train==0).sum()} ({(y_train==0).sum()/len(y_train)*100:.1f}%)")

print(f"\nClass distribution in test set:")
print(f"  Positive (1):  {(y_test==1).sum()} ({(y_test==1).sum()/len(y_test)*100:.1f}%)")
print(f"  Negative (0):  {(y_test==0).sum()} ({(y_test==0).sum()/len(y_test)*100:.1f}%)")

Train Test Split Results:
  Training set:   746 samples
  Test set:       187 samples
  Split ratio:    80% train / 20% test

Class distribution in training set:
  Positive (1):  337 (45.2%)
  Negative (0):  409 (54.8%)

Class distribution in test set:
  Positive (1):  85 (45.5%)
  Negative (0):  102 (54.5%)


### Observations

The dataset was split into 746 training samples and 187 test
samples. The stratification worked correctly — both sets maintain
approximately the same class proportions as the original dataset,
with around 45% positive and 55% negative samples in each split.
The test set will not be used during preprocessing or training
and will only be used for final model evaluation.

## 3. Preprocessing Pipeline

All 8100 HOG features are numerical, so a single numeric pipeline
is required. The pipeline applies StandardScaler to standardise
the features to zero mean and unit variance.

Standardisation is important for SVM in particular, which is
sensitive to the scale of input features. Without scaling,
features with larger values would dominate the decision boundary
and produce poor results.

The pipeline is fitted on the training data only. The fitted
pipeline is then used to transform both the training and test
sets, ensuring no information from the test set influences the
preprocessing.

In [ ]:
# Define numeric features — all 8100 HOG features are numeric
numeric_features = list(range(X_train.shape[1]))

# Build numeric pipeline
numeric_pipeline = Pipeline(steps=[
    ('scaler', StandardScaler())
])

# Display pipeline
print("Numeric Pipeline:")
print(numeric_pipeline)

# Build ColumnTransformer
preprocessor = ColumnTransformer(transformers=[
    ('numeric', numeric_pipeline, numeric_features)
])

print("\nColumn Transformer:")
print(preprocessor)

Numeric Pipeline:
Pipeline(steps=[('scaler', StandardScaler())])

Column Transformer:
ColumnTransformer(transformers=[('numeric',
                                 Pipeline(steps=[('scaler', StandardScaler())]),
                                 [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,
                                  14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24,
                                  25, 26, 27, 28, 29, ...])])


### Observations

The numeric pipeline contains a single StandardScaler step applied
to all 8100 HOG features. The ColumnTransformer wraps this pipeline
and applies it to all feature columns. No nominal pipeline is needed
since all HOG features are continuous numerical values with no
categorical variables in the features matrix.

## 4. Fit and Transform

We fit the preprocessor on the training data only and use it
to transform both the training and test sets. The transformed
data is saved for use in the modelling notebooks.

In [ ]:
# Fit on training data only — prevents data leakage
preprocessor.fit(X_train)

# Transform both sets
X_train_processed = preprocessor.transform(X_train)
X_test_processed  = preprocessor.transform(X_test)

print("Preprocessing complete")
print(f"\nX_train_processed shape: {X_train_processed.shape}")
print(f"X_test_processed shape:  {X_test_processed.shape}")

# Verify scaling — mean should be ~0 and std ~1 on training set
print(f"\nVerification on training set (first 5 features):")
print(f"  Mean: {X_train_processed[:, :5].mean(axis=0).round(4)}")
print(f"  Std:  {X_train_processed[:, :5].std(axis=0).round(4)}")

# Show head of processed training data
X_train_df = pd.DataFrame(
    X_train_processed[:, :8],
    columns=[f'HOG_{i}' for i in range(8)]
)
print(f"\nX_train_processed head (first 8 features):")
print(X_train_df.head())

X_test_df = pd.DataFrame(
    X_test_processed[:, :8],
    columns=[f'HOG_{i}' for i in range(8)]
)
print(f"\nX_test_processed head (first 8 features):")
print(X_test_df.head())

Preprocessing complete

X_train_processed shape: (746, 8100)
X_test_processed shape:  (187, 8100)

Verification on training set (first 5 features):
  Mean: [-0.  0. -0.  0. -0.]
  Std:  [1. 1. 1. 1. 1.]

X_train_processed head (first 8 features):
      HOG_0     HOG_1     HOG_2     HOG_3     HOG_4     HOG_5     HOG_6  \
0 -1.276127 -0.164821  1.137702 -0.683583 -1.355305 -0.494011 -0.738571   
1  0.798884  0.120531 -0.014812 -0.456812  1.036340 -0.281269  0.439909   
2  0.634700 -1.277916 -1.495524 -1.071538 -1.152496 -0.512500 -0.660449   
3  1.807412 -1.277916 -1.456294 -1.080707 -1.100771 -0.672426 -0.627647   
4  0.651925 -1.111840  1.237212  2.400387  1.606089  0.163505 -0.556631   

      HOG_7  
0 -0.367733  
1  0.699106  
2  3.360094  
3  0.152978  
4 -0.429388  

X_test_processed head (first 8 features):
      HOG_0     HOG_1     HOG_2     HOG_3     HOG_4     HOG_5     HOG_6  \
0  0.794541 -1.240769  0.329350 -1.055426  0.239888 -0.672426 -0.334114   
1 -1.091537 -1.210603 -0.

### Observations

The preprocessor was fitted on the training data only and then
used to transform both sets. The verification confirms that the
training features have a mean of approximately 0 and standard
deviation of 1 across all features, confirming that StandardScaler
worked correctly. The test set was transformed using the same
scaler fitted on training data, ensuring no data leakage occurred.

In [ ]:
import joblib

# Save processed arrays
np.save(f'{SAVE_DIR}/X_train_processed.npy', X_train_processed)
np.save(f'{SAVE_DIR}/X_test_processed.npy',  X_test_processed)
np.save(f'{SAVE_DIR}/y_train.npy',            y_train)
np.save(f'{SAVE_DIR}/y_test.npy',             y_test)

# Save fitted preprocessor for future use
joblib.dump(preprocessor, f'{SAVE_DIR}/preprocessor.pkl')

print("Saved files:")
print(f"  X_train_processed.npy — {X_train_processed.shape}")
print(f"  X_test_processed.npy  — {X_test_processed.shape}")
print(f"  y_train.npy           — {y_train.shape}")
print(f"  y_test.npy            — {y_test.shape}")
print(f"  preprocessor.pkl      — fitted StandardScaler pipeline")

print(f"""
PREPROCESSING SUMMARY
  Total samples:        933
  Training samples:     746  (80%)
  Test samples:         187  (20%)
  Features per sample:  8100
  Split:                Stratified — class proportions preserved
  Scaling:              StandardScaler — zero mean, unit variance
  Data leakage:         None — scaler fitted on training data only

PREPROCESSING COMPLETE — Proceed to 04_svm_classifier.ipynb
""")

Saved files:
  X_train_processed.npy — (746, 8100)
  X_test_processed.npy  — (187, 8100)
  y_train.npy           — (746,)
  y_test.npy            — (187,)
  preprocessor.pkl      — fitted StandardScaler pipeline

PREPROCESSING SUMMARY
  Total samples:        933
  Training samples:     746  (80%)
  Test samples:         187  (20%)
  Features per sample:  8100
  Split:                Stratified — class proportions preserved
  Scaling:              StandardScaler — zero mean, unit variance
  Data leakage:         None — scaler fitted on training data only

PREPROCESSING COMPLETE — Proceed to 04_svm_classifier.ipynb



### Summary

The preprocessing pipeline is complete. All 933 HOG feature
vectors were standardised to zero mean and unit variance using
StandardScaler. The train test split was performed before fitting
the scaler to prevent data leakage. The processed training set
contains 746 samples and the test set contains 187 samples, with
class proportions preserved in both sets. All files have been
saved to Google Drive and are ready for model training.